In [68]:
%env XLA_PYTHON_CLIENT_PREALLOCATE=False
import jax.numpy as jnp
import pickle
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# =============================
# Load all PKL files
# =============================
DIR = "./"
results_list = []
titles_list = []
eps_list = []

for file in os.listdir(DIR):
    if file.endswith(".pkl"):
        # Load pickle
        with open(os.path.join(DIR, file), "rb") as f:
            my_pkl = pickle.load(f)

        # Append 5th element of 'res'
        results_list.append(my_pkl['res'][4])
        titles_list.append(file)

        # Parse eps from filename: assumes 'eps0.1000' style
        start = file.find("eps")
        end = file.find("_", start)
        eps_val = float(file[start+3:end])
        eps_list.append(eps_val)

# =============================
# Build pandas DataFrame
# =============================
df = pd.DataFrame({
    "filename": titles_list,
    "eps": eps_list,
    "res5": results_list
})

# Sort by eps for convenience
df = df.sort_values("eps").reset_index(drop=True)

print(df.head())

# =============================
# Example: iterate over eps and do something
# =============================
for eps_val, group in df.groupby("eps"):
    # For illustration: print shape of the 5th result
    print(f"eps={eps_val}: number of entries = {len(group)}, shapes = {[r.shape if hasattr(r, 'shape') else None for r in group['res5']]}")


# =============================
# Concatenate results for each epsilon
# =============================
# Store as new column 'res5_concat' for each epsilon
df_concat = (
    df.groupby("eps")["res5"]
      .apply(lambda x: np.concatenate(x.values, axis=0))
      .reset_index()
      .rename(columns={"res5": "res5_concat"})
)


env: XLA_PYTHON_CLIENT_PREALLOCATE=False
                           filename   eps  \
0  1__eps0.1000_1.py_0311143228.pkl  0.10   
1  1__eps0.1000_1.py_0311143239.pkl  0.10   
2  1__eps0.1000_1.py_0311143217.pkl  0.10   
3  1__eps0.1000_1.py_0311143249.pkl  0.10   
4  1__eps0.2500_1.py_0311143303.pkl  0.25   

                                                res5  
0  [[-2.1191287, -1.4118745, -0.21753001, 0.0, 0....  
1  [[-2.1176033, -1.42824, -0.22111715, 0.0, 0.0,...  
2  [[-2.1156154, -1.4081048, -0.21414246, 0.0, 0....  
3  [[-2.132216, -1.4217522, -0.19291364, 0.0, 0.0...  
4  [[-2.1248238, -1.4210904, -0.21842259, 0.0, 0....  
eps=0.1: number of entries = 4, shapes = [(10, 6), (10, 6), (10, 6), (10, 6)]
eps=0.25: number of entries = 4, shapes = [(10, 6), (10, 6), (10, 6), (10, 6)]


In [69]:
def rel_error(x, y):
    return jnp.abs((x - y) / y)


def computeZ_from_data(my_pkl):
    logZoverZs = my_pkl['res'][4]
    ZoverZs = jnp.exp(logZoverZs)
    Z = jnp.prod(ZoverZs, axis=-1)
    return Z


def Z_med(logZoverZs):
    """
    Given a sequence of logZoverZs of log of Monte-Carlo estimates of ratio
    of normalising constant, compute Z^med given by the product of the median
    estimates for each ratio of normalising constant
    """
    ZoverZs = jnp.exp(logZoverZs)
    medians = jnp.median(ZoverZs, axis=-1)
    return jnp.prod(medians, axis=-1)


def concatenate_logZoverZs(my_pkls, T):
    number_of_pkls = len(my_pkls)
    in_parallel = 1
    if my_pkls:
        in_parallel = my_pkls[0]['res'][4].shape[0]
    logZoverZs = np.zeros(shape=(number_of_pkls, in_parallel, T))
    for idx, pkl in enumerate(my_pkls):
        logZoverZs[idx] = pkl['res'][4]
    logZoverZs = logZoverZs.reshape((-1, T))
    return logZoverZs


In [70]:
true_Z = jnp.exp(my_pkl['config']['logZ'])

In [71]:

# =============================
# Compute jnp.prod(jnp.exp(...), axis=-1) for each epsilon
# =============================
df_concat["prod_exp"] = df_concat["res5_concat"].apply(lambda arr: jnp.prod(jnp.exp(arr), axis=-1))
df_concat["rel_error"] = df_concat["prod_exp"].apply(lambda x: rel_error(x, true_Z))
# =============================
# Now df_concat contains
# eps | res5_concat | prod_exp
# =============================
print(df_concat)

    eps                                        res5_concat  \
0  0.10  [[-2.1191287, -1.4118745, -0.21753001, 0.0, 0....   
1  0.25  [[-2.1248238, -1.4210904, -0.21842259, 0.0, 0....   

                                            prod_exp  \
0  [0.023552263, 0.023569856, 0.023300493, 0.0237...   
1  [0.023182983, 0.023510972, 0.024292247, 0.0248...   

                                           rel_error  
0  [0.012526493, 0.013282813, 0.001702742, 0.0220...  
1  [0.0033491103, 0.010751361, 0.044338875, 0.066...  


In [72]:
df_concat

,eps,res5_concat,prod_exp,rel_error
0,0.10,"[[-2.1191287, -1.4118745, -0.21753001, 0.0, 0....","[0.023552263, 0.023569856, 0.023300493, 0.0237...","[0.012526493, 0.013282813, 0.001702742, 0.0220..."
1,0.25,"[[-2.1248238, -1.4210904, -0.21842259, 0.0, 0....","[0.023182983, 0.023510972, 0.024292247, 0.0248...","[0.0033491103, 0.010751361, 0.044338875, 0.066..."


In [73]:
# Assume df_concat has columns:
# eps | res5_concat | prod_exp | prod_exp_f
# And each 'res5_concat' is an array of shape (K, T)
J = 4  # number of splits per epsilon

# Prepare a new list to build the expanded DataFrame
expanded_records = []

for _, row in df_concat.iterrows():
    eps = row["eps"]
    res5 = row["res5_concat"]
    prod_exp = row["prod_exp"]
    prod_exp_f = row["rel_error"]

    # Compute chunk sizes
    K = res5.shape[0]
    split_size = K // J
    remainder = K % J  # if not divisible exactly

    # Use np.array_split to handle uneven splits automatically
    res5_splits = np.array_split(np.array(res5), J, axis=0)
    prod_exp_splits = np.array_split(np.array(prod_exp), J, axis=0)
    prod_exp_f_splits = np.array_split(np.array(prod_exp_f), J, axis=0)

    # Append each split as a new row
    for i in range(J):
        expanded_records.append({
            "eps": eps,
            "res5_split": res5_splits[i],
            "prod_exp_split": prod_exp_splits[i],
            "rel_error_split": prod_exp_f_splits[i]
        })

# Build new DataFrame
df_splits = pd.DataFrame(expanded_records)

print(df_splits.head())
print(f"Total rows after splitting: {len(df_splits)}")

    eps                                         res5_split  \
0  0.10  [[-2.1191287, -1.4118745, -0.21753001, 0.0, 0....   
1  0.10  [[-2.1176033, -1.42824, -0.22111715, 0.0, 0.0,...   
2  0.10  [[-2.1156154, -1.4081048, -0.21414246, 0.0, 0....   
3  0.10  [[-2.132216, -1.4217522, -0.19291364, 0.0, 0.0...   
4  0.25  [[-2.1248238, -1.4210904, -0.21842259, 0.0, 0....   

                                      prod_exp_split  \
0  [0.023552263, 0.023569856, 0.023300493, 0.0237...   
1  [0.023122238, 0.023515543, 0.023303451, 0.0231...   
2  [0.023804933, 0.023373798, 0.023012279, 0.0231...   
3  [0.023591192, 0.023416072, 0.022742765, 0.0234...   
4  [0.023182983, 0.023510972, 0.024292247, 0.0248...   

                                     rel_error_split  
0  [0.012526493, 0.013282813, 0.001702742, 0.0220...  
1  [0.0059605576, 0.010947868, 0.0018299031, 0.00...  
2  [0.023388918, 0.004854144, 0.010687781, 0.0058...  
3  [0.014200087, 0.006671555, 0.022274338, 0.0080...  
4  [0.003349110

In [74]:
df_splits["Z_med_rel_error"] = df_splits["res5_split"].apply(lambda x: rel_error(Z_med(x.T), true_Z))

In [75]:
"""
We will compute:

Per chunk: prob_ge_eps_chunk = fraction of rel_error_split >= eps

Per epsilon: Z_med_rel_error_agg = fraction of all rel_error_split in all chunks of that epsilon >= eps
"""



# -----------------------------
# 1️⃣ Per-chunk probability
# -----------------------------
def compute_prob_ge_eps(row):
    # fraction of values in rel_error_split >= eps of the row
    return float(jnp.mean(row["rel_error_split"] >= row["eps"]))

df_splits["failure_prob_chunk"] = df_splits.apply(compute_prob_ge_eps, axis=1)

# -----------------------------
# 2️⃣ Per-epsilon aggregated probability
# -----------------------------
df_splits["failure_prob"] = np.nan  # initialize

for eps_val, group in df_splits.groupby("eps"):
    # Concatenate all rel_error_split arrays for this epsilon
    all_rel_error = np.concatenate(group["rel_error_split"].values, axis=0)
    prob_agg = float(jnp.mean(all_rel_error >= eps_val))

    # Store only in the first row of this epsilon
    first_idx = group.index[0]
    df_splits.at[first_idx, "failure_prob"] = prob_agg

In [76]:
df_splits

,eps,res5_split,prod_exp_split,rel_error_split,Z_med_rel_error,failure_prob_chunk,failure_prob
0,0.10,"[[-2.1191287, -1.4118745, -0.21753001, 0.0, 0....","[0.023552263, 0.023569856, 0.023300493, 0.0237...","[0.012526493, 0.013282813, 0.001702742, 0.0220...",0.0017055447,0.0,0.0
1,0.10,"[[-2.1176033, -1.42824, -0.22111715, 0.0, 0.0,...","[0.023122238, 0.023515543, 0.023303451, 0.0231...","[0.0059605576, 0.010947868, 0.0018299031, 0.00...",0.003677503,0.0,NaN
2,0.10,"[[-2.1156154, -1.4081048, -0.21414246, 0.0, 0....","[0.023804933, 0.023373798, 0.023012279, 0.0231...","[0.023388918, 0.004854144, 0.010687781, 0.0058...",0.002576214,0.0,NaN
3,0.10,"[[-2.132216, -1.4217522, -0.19291364, 0.0, 0.0...","[0.023591192, 0.023416072, 0.022742765, 0.0234...","[0.014200087, 0.006671555, 0.022274338, 0.0080...",0.003986838,0.0,NaN
4,0.25,"[[-2.1248238, -1.4210904, -0.21842259, 0.0, 0....","[0.023182983, 0.023510972, 0.024292247, 0.0248...","[0.0033491103, 0.010751361, 0.044338875, 0.066...",0.01144282,0.0,0.0
5,0.25,"[[-2.106836, -1.4112632, -0.22498424, 0.0, 0.0...","[0.023680968, 0.023768853, 0.022826247, 0.0219...","[0.018059602, 0.021837842, 0.018685399, 0.0582...",0.0045905327,0.0,NaN
6,0.25,"[[-2.1239476, -1.4818444, -0.21457888, 0.0, 0....","[0.02191967, 0.023589741, 0.02377125, 0.024832...","[0.057659723, 0.014137707, 0.0219409, 0.067546...",0.010744475,0.0,NaN
7,0.25,"[[-2.1233406, -1.4492433, -0.19559897, 0.0, 0....","[0.02309399, 0.023358153, 0.024375036, 0.02258...","[0.007174995, 0.0041815834, 0.047898024, 0.029...",0.005139696,0.0,NaN


In [ ]:
f